In [ ]:
"""=============================================================================商品组合分析：波士顿矩阵 + 聚类分析 + 方法论改进=============================================================================原始项目: 和鲸社区 yz7zzxj001来源: https://www.heywhale.com/mw/project/5f1f24d9d278b1002c200e40本脚本:  Part A — 完整复现原始分析  Part B — 方法论改进 (年度数据 + 相对市场份额 + 气泡矩阵 + 严格聚类评估)============================================================================="""import numpy as npimport pandas as pdimport seaborn as snsimport matplotlibmatplotlib.use('Agg')  # 非交互式后端, 避免 plt.show() 阻塞import matplotlib.pyplot as pltfrom matplotlib.patches import FancyBboxPatchimport warningswarnings.filterwarnings('ignore')# ============================================================# 全局绘图设置# ============================================================plt.rcParams['font.sans-serif'] = ['SimHei']plt.rcParams['axes.unicode_minus'] = Falsesns.set(font='SimHei', style='whitegrid')plt.rcParams['figure.dpi'] = 120plt.rcParams['savefig.dpi'] = 150

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗

In [ ]:
# ║                      Part A: 原始分析复现                                  ║

In [ ]:
# ╚═══════════════════════════════════════════════════════════════════════════╝print("=" * 70)print("Part A: 原始分析复现")print("=" * 70)

In [ ]:
# ── A1. 读取数据 ──────────────────────────────────────────────────────────print("\n── A1. 读取数据 ──")data = pd.read_excel('某厂商旗舰店脱敏数据.xlsx', header=0, index_col=0)print(f"数据集规模: {data.shape[0]} 产品 × {data.shape[1]} 字段")print(data.head(8))

In [ ]:
# ── A2. 特征衍生 ──────────────────────────────────────────────────────────print("\n── A2. 特征衍生 ──")data['预估交易金额'] = data['预估月销量'] * data['价格']data['预估交易占比'] = data['预估交易金额'] / data['预估交易金额'].sum()data['专区'] = data['名称'].str.contains('专区').astype(int)print("预估交易增幅分布:")print(data['预估交易增幅'].describe(    percentiles=[i / 100 for i in range(0, 100, 5)]))# 异常值检查 (原文: 增幅 > 1.1)outliers = data[data['预估交易增幅'] > 1.1]print(f"\n高增幅异常值 ({len(outliers)} 个):")print(outliers[['名称', '价格', '总销量', '预估月销量', '6月销量', '预估交易增幅']])# 专区 vs 非专区 对比zone_stats = data.groupby('专区').agg({    '预估交易增幅': 'describe',    '预估交易占比': 'sum'})print("\n专区 vs 非专区:")print(data.groupby('专区')['预估交易增幅'].describe())print(f"\n专区交易占比合计: {data[data['专区']==1]['预估交易占比'].sum():.4%}")print(f"非专区交易占比合计: {data[data['专区']==0]['预估交易占比'].sum():.4%}")

In [ ]:
# ── A3. 专区商品聚类分析 ──────────────────────────────────────────────────print("\n── A3. 专区商品聚类分析 ──")zhuanqu = data[data['专区'] == 1].copy()changgui = data[data['专区'] == 0].copy()from sklearn.cluster import KMeans, SpectralClusteringfrom sklearn.preprocessing import MinMaxScaler, StandardScalerfrom sklearn.metrics import silhouette_score as ss# 特征选择print("\n专区商品相关性矩阵:")zhuanqu_numeric = zhuanqu.select_dtypes(include=[np.number])print(zhuanqu_numeric.corr().round(4))# 准备聚类特征x_raw = zhuanqu[['价格', '总销量', '评价数', '预估交易占比', '预估交易增幅']].copy()# 归一化: 前4列 MinMax, 增幅 StandardScalerx_scaled = x_raw.copy()for col in ['价格', '总销量', '评价数', '预估交易占比']:    x_scaled[[col]] = MinMaxScaler().fit_transform(x_raw[[col]])x_scaled[['预估交易增幅']] = StandardScaler().fit_transform(x_raw[['预估交易增幅']])

In [ ]:
# ── KMeans 尝试 ──print("\n--- KMeans 聚类 ---")k_list = list(range(2, 15))kmeans_scores = {}for k in k_list:    km = KMeans(n_clusters=k, random_state=42, n_init=10)    labels = km.fit_predict(x_scaled)    kmeans_scores[k] = ss(x_scaled, labels, metric='euclidean')best_k_kmeans = max(kmeans_scores, key=kmeans_scores.get)print(f"KMeans 最佳 K={best_k_kmeans}, 轮廓系数={kmeans_scores[best_k_kmeans]:.4f}")print("  → 原文发现 K=2 时轮廓系数最高 (~0.75), 实际上是分开离群点")# 对集群点二次聚类cluster_labels = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(x_scaled)x_cluster = x_scaled[cluster_labels == 0].copy()cluster_scores = {}for k in k_list:    km = KMeans(n_clusters=k, random_state=42, n_init=10)    labels = km.fit_predict(x_cluster)    cluster_scores[k] = ss(x_cluster, labels, metric='euclidean')best_k_cluster = max(cluster_scores, key=cluster_scores.get)print(f"排除离群点后 KMeans 最佳 K={best_k_cluster}, 轮廓系数={cluster_scores[best_k_cluster]:.4f}")print("  → 原文: K=8时轮廓系数最大(~0.475), K=5也可接受")

In [ ]:
# ── 谱聚类尝试 (原文推荐) ──print("\n--- 谱聚类 ---")spectral_scores = {}for k in k_list:    sc = SpectralClustering(n_clusters=k, random_state=42,                            affinity='rbf', assign_labels='kmeans')    labels = sc.fit_predict(x_scaled)    spectral_scores[k] = ss(x_scaled, labels, metric='euclidean')best_k_sc = max(spectral_scores, key=spectral_scores.get)print(f"谱聚类 最佳 K={best_k_sc}, 轮廓系数={spectral_scores[best_k_sc]:.4f}")print("  → 原文: 次高点 K=3, 实际也是分开离群点")# 排除离群点后谱聚类x_no_outlier = x_scaled[cluster_labels == 0].copy()spectral_scores_clean = {}for k in k_list:    sc = SpectralClustering(n_clusters=k, random_state=42,                            affinity='rbf', assign_labels='kmeans')    labels = sc.fit_predict(x_no_outlier)    spectral_scores_clean[k] = ss(x_no_outlier, labels, metric='euclidean')best_k_scc = max(spectral_scores_clean, key=spectral_scores_clean.get)print(f"排除离群点后谱聚类 最佳 K={best_k_scc}, 轮廓系数={spectral_scores_clean[best_k_scc]:.4f}")print("  → 原文: K=5 较合适")

In [ ]:
# ── 最终聚类 (K=7, 原文选择) ──print("\n--- 最终聚类结果 (K=7) ---")final_sc = SpectralClustering(n_clusters=7, random_state=42,                               affinity='rbf', assign_labels='kmeans')zhuanqu['cluster'] = final_sc.fit_predict(x_scaled)print("各簇产品数:")print(zhuanqu.groupby('cluster').size())print("\n各簇中位数特征:")cluster_median = zhuanqu.groupby('cluster')[['价格', '总销量', '评价数',                                               '预估交易占比', '预估交易增幅']].median()print(cluster_median.to_string())# 雷达图 (为代表性簇绘制)# 选择非离群簇 (size > 1)valid_clusters = zhuanqu.groupby('cluster').size()valid_clusters = valid_clusters[valid_clusters > 2].index.tolist()fig, axes = plt.subplots(1, min(3, len(valid_clusters)),                          figsize=(15, 5), subplot_kw=dict(projection='polar'))if len(valid_clusters) == 1:    axes = [axes]for idx, cluster_id in enumerate(valid_clusters[:3]):    ax = axes[idx]    subset = zhuanqu[zhuanqu['cluster'] == cluster_id]    radar_data = subset[['价格', '总销量', '评价数',                          '预估交易占比', '预估交易增幅']].apply(        lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8)    ).mean()    labels = radar_data.index.tolist()    values = radar_data.values.tolist()    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()    values += values[:1]    angles += angles[:1]    ax.fill(angles, values, alpha=0.3, color='steelblue')    ax.plot(angles, values, 'o-', color='steelblue', linewidth=2)    ax.set_xticks(angles[:-1])    ax.set_xticklabels(labels, fontsize=10)    ax.set_title(f'簇 {cluster_id} 特征雷达图\n(n={len(subset)})', fontsize=12, pad=20)    ax.set_ylim(0, 1)plt.suptitle('专区商品聚类雷达图', fontsize=14, y=1.02)plt.tight_layout()plt.savefig('专区聚类雷达图.png', bbox_inches='tight', dpi=150)plt.show()print("\n 聚类完成, 雷达图已保存至: 专区聚类雷达图.png")

In [ ]:
# ── A4. 非专区波士顿矩阵 ────────────────────────────────────────────────print("\n── A4. 非专区: 波士顿矩阵分析 ──")changgui = data[data['专区'] == 0].copy()mean_growth = changgui['预估交易增幅'].mean()mean_share = changgui['预估交易占比'].mean()print(f"非专区均值线 → 交易增幅: {mean_growth:.4f}, 交易占比: {mean_share:.4f} ({mean_share:.2%})")# 四象限分类conditions = [    (changgui['预估交易增幅'] > mean_growth) & (changgui['预估交易占比'] > mean_share),    (changgui['预估交易增幅'] <= mean_growth) & (changgui['预估交易占比'] > mean_share),    (changgui['预估交易增幅'] > mean_growth) & (changgui['预估交易占比'] <= mean_share),    (changgui['预估交易增幅'] <= mean_growth) & (changgui['预估交易占比'] <= mean_share),]choices = ['爆款(Star)', '金牛(Cash Cow)', '潜力款(Question Mark)', '淘汰款(Dog)']changgui['BCG'] = np.select(conditions, choices, default='未分类')print("\n波士顿矩阵分类结果:")bcg_summary = changgui.groupby('BCG').agg(    产品数=('名称', 'count'),    交易占比合计=('预估交易占比', 'sum'),    平均增幅=('预估交易增幅', 'mean'),    交易金额=('预估交易金额', 'sum'),).sort_values('交易金额', ascending=False)print(bcg_summary.to_string())# 波士顿矩阵散点图fig, ax = plt.subplots(figsize=(12, 8))colors = {'爆款(Star)': '#e74c3c', '金牛(Cash Cow)': '#f39c12',          '潜力款(Question Mark)': '#2ecc71', '淘汰款(Dog)': '#95a5a6'}for label in choices:    subset = changgui[changgui['BCG'] == label]    sizes = np.sqrt(subset['总销量']) / 10  # 气泡大小代表总销量    ax.scatter(subset['预估交易增幅'], subset['预估交易占比'],               s=sizes, c=colors[label], alpha=0.7, edgecolors='white',               linewidth=0.5, label=f"{label} (n={len(subset)})")# 标注关键产品key_products = ['常态3', '常态16', 'ZAM推荐常态12', '常态4', '常态0', '常态11']for _, row in changgui.iterrows():    if row['名称'] in key_products:        offset = (15, 15) if 'ZAM' in row['名称'] else (10, -10)        ax.annotate(row['名称'], (row['预估交易增幅'], row['预估交易占比']),                     textcoords="offset points", xytext=offset,                     fontsize=9, arrowprops=dict(arrowstyle='->', alpha=0.5))ax.axhline(mean_share, color='gray', linestyle='--', alpha=0.5)ax.axvline(mean_growth, color='gray', linestyle='--', alpha=0.5)ax.set_xlabel('预估交易增幅', fontsize=12)ax.set_ylabel('预估交易占比', fontsize=12)ax.set_title('波士顿矩阵: 非专区商品结构\n(气泡大小=总销量)', fontsize=14)ax.legend(loc='upper left', fontsize=10)plt.tight_layout()plt.savefig('波士顿矩阵_非专区.png', bbox_inches='tight', dpi=150)plt.show()print('  → 散点图已保存至: 波士顿矩阵_非专区.png')

In [ ]:
# ── A5. 价格段交叉分析 ─────────────────────────────────────────────────print("\n── A5. 价格段交叉分析 ──")price_bins = [-np.inf, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, np.inf]price_labels = ['≤10', '(10,20]', '(20,30]', '(30,40]', '(40,50]',                '(50,60]', '(60,70]', '(70,80]', '(80,90]', '(90,100]', '>100']bst = changgui.copy()bst['价格段'] = pd.cut(bst['价格'], bins=price_bins, labels=price_labels)# 价格段 × BCG 交叉表cross_analysis = bst.groupby(['价格段', 'BCG'], observed=False).agg(    产品数=('名称', 'count'),    总销量=('总销量', 'sum'),    预估月销量=('预估月销量', 'sum'),    平均增幅=('预估交易增幅', 'mean'),    交易占比合计=('预估交易占比', 'sum'),    交易金额=('预估交易金额', 'sum'),).dropna()# 计算单品平均交易占比cross_analysis['单品均交易占比'] = (cross_analysis['交易占比合计'] /                                  cross_analysis['产品数'])print("\n价格段 × BCG 交叉分析:")print(cross_analysis.to_string())# 热力图pivot_table = cross_analysis.reset_index().pivot_table(    values='产品数', index='价格段', columns='BCG', fill_value=0, observed=False)fig, ax = plt.subplots(figsize=(10, 6))sns.heatmap(pivot_table.astype(int), annot=True, fmt='d', cmap='YlOrRd',            ax=ax, cbar_kws={'label': '产品数'})ax.set_title('价格段 × 波士顿矩阵: 产品数分布', fontsize=14)plt.tight_layout()plt.savefig('价格段_BCG_热力图.png', bbox_inches='tight', dpi=150)plt.show()print('  → 热力图已保存至: 价格段_BCG_热力图.png')

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗

In [ ]:
# ║                   Part B: 方法论改进                                      ║

In [ ]:
# ╚═══════════════════════════════════════════════════════════════════════════╝print("\n\n" + "=" * 70)print("Part B: 方法论改进")print("=" * 70)print("""改进要点:  1. 月度数据 → 模拟年度数据 (更符合BCG矩阵的原始定义)  2. 交易额占比 → 相对市场份额 (vs 最大竞争者, BCG矩阵的标准维度)  3. 增幅 → 市场增长率 (基于品类整体增长, 而非自身环比)  4. 二维 → 三维气泡矩阵 (气泡大小=销售额, 增加信息密度)  5. 单一聚类方法 → 多方法对比 + Gap Statistic + 稳定性评估  6. 增加产品生命周期阶段自动判定""")

In [ ]:
# ── B1. 模拟年度数据 ───────────────────────────────────────────────────print("── B1. 构建年度面板数据 ──")np.random.seed(42)n_products = len(data)products = data['名称'].values# 构建12个月的面板数据months = pd.date_range('2025-01-01', periods=12, freq='ME')panel_data = []for i, (_, row) in enumerate(data.iterrows()):    base_monthly = row['预估月销量']    base_price = row['价格']    is_zone = 1 if '专区' in row['名称'] else 0    growth_rate = row['预估交易增幅']    # 模拟各月销量: 基于原月销量 + 月度波动 + 趋势    # 安全处理 base_monthly = 0 的情况    if base_monthly > 0:        trend = np.linspace(0, growth_rate * base_monthly, 12)        seasonality = 0.15 * base_monthly * np.sin(np.linspace(0, 2 * np.pi, 12))    else:        trend = np.zeros(12)        seasonality = np.zeros(12)    # 不同产品类型有不同的波动特征    if '常态' in row['名称'] and growth_rate > 0.5:        # 爆款: 快速增长        monthly_sales = base_monthly * (1 + np.linspace(0, growth_rate, 12))        noise = np.random.normal(0, 0.05, 12) * max(base_monthly, 1)    elif is_zone:        # 专区: 销量小, 波动大        if base_monthly > 0:            ratio = trend / (base_monthly + 1e-8)        else:            ratio = np.zeros(12)        monthly_sales = np.maximum(0, base_monthly * (1 + ratio * 0.3)                                    + seasonality * 0.8)        noise = np.random.normal(0, max(1, base_monthly * 0.2), 12)    else:        # 常规产品        if base_monthly > 0:            ratio = trend / (base_monthly + 1e-8)        else:            ratio = np.zeros(12)        monthly_sales = base_monthly * (1 + ratio * 0.5)        noise = np.random.normal(0, max(1, base_monthly * 0.08), 12)    monthly_sales = np.maximum(0, monthly_sales + noise)    for m_idx, month in enumerate(months):        panel_data.append({            '名称': row['名称'],            'month': month,            '价格': base_price,            '月销量': max(0, int(monthly_sales[m_idx])),            'is_zone': is_zone,        })panel_df = pd.DataFrame(panel_data)panel_df['月销售额'] = panel_df['月销量'] * panel_df['价格']print(f"面板数据: {panel_df.shape[0]} 条记录 (76 产品 × 12 月)")print(panel_df.head(10))

In [ ]:
# ── B2. 年度指标计算 ───────────────────────────────────────────────────print("\n── B2. 年度BCG指标计算 ──")# 汇总到年度级别annual = panel_df.groupby('名称').agg(    年销售额=('月销售额', 'sum'),    月均销量=('月销量', 'mean'),    价格=('价格', 'first'),    is_zone=('is_zone', 'first'),)# 计算年增长率 (上半年 vs 下半年)h1 = panel_df[panel_df['month'] <= '2025-06-30'].groupby('名称')['月销售额'].sum()h2 = panel_df[panel_df['month'] > '2025-06-30'].groupby('名称')['月销售额'].sum()annual['年增长率'] = np.where(h1 > 0, (h2 - h1) / h1, 0)# 相对市场份额: 该产品销售额 / 最大竞争者(非本品类)销售额# 按价格段分组, 计算价格段内的相对份额annual['价格段'] = pd.cut(annual['价格'], bins=price_bins, labels=price_labels)annual['相对市场份额'] = 0.0for seg in annual['价格段'].unique():    mask = annual['价格段'] == seg    if mask.sum() > 1:        max_sales = annual.loc[mask, '年销售额'].max()        annual.loc[mask, '相对市场份额'] = (            annual.loc[mask, '年销售额'] / max_sales        )    elif mask.sum() == 1:        annual.loc[mask, '相对市场份额'] = 1.0# 品类增长率: 该价格段整体的增长率category_growth = annual.groupby('价格段')['年增长率'].median()annual['品类增长率'] = annual['价格段'].map(category_growth)annual['相对增长率'] = annual['年增长率'] - annual['品类增长率']print("年度指标摘要:")print(annual.describe().round(4).to_string())

In [ ]:
# ── B3. 改进波士顿矩阵 ────────────────────────────────────────────────print("\n── B3. 改进波士顿矩阵 (相对市场份额 × 市场增长率) ──")non_zone_annual = annual[annual['is_zone'] == 0].copy()# 新的分割线median_share = non_zone_annual['相对市场份额'].median()median_growth = non_zone_annual['品类增长率'].median()cond_improved = [    (non_zone_annual['相对市场份额'] > median_share) & (non_zone_annual['品类增长率'] > median_growth),    (non_zone_annual['相对市场份额'] > median_share) & (non_zone_annual['品类增长率'] <= median_growth),    (non_zone_annual['相对市场份额'] <= median_share) & (non_zone_annual['品类增长率'] > median_growth),    (non_zone_annual['相对市场份额'] <= median_share) & (non_zone_annual['品类增长率'] <= median_growth),]non_zone_annual['BCG_Improved'] = np.select(cond_improved, choices, default='未分类')# 改进版散点图 (三维气泡)fig, ax = plt.subplots(figsize=(14, 9))for label in choices:    subset = non_zone_annual[non_zone_annual['BCG_Improved'] == label]    if len(subset) == 0:        continue    # 气泡大小 = 年销售额    sizes = np.sqrt(subset['年销售额']) / 30    ax.scatter(subset['相对增长率'], subset['相对市场份额'],               s=sizes, c=colors[label], alpha=0.7, edgecolors='white',               linewidth=0.8, label=f"{label} (n={len(subset)})")# 标注for _, row in non_zone_annual.iterrows():    if row['年销售额'] > non_zone_annual['年销售额'].quantile(0.85):        ax.annotate(row.name, (row['相对增长率'], row['相对市场份额']),                     fontsize=8, alpha=0.8,                     xytext=(5, 5), textcoords="offset points")ax.axhline(median_share, color='gray', linestyle='--', alpha=0.5,            label=f'相对份额中位数: {median_share:.2f}')ax.axvline(0, color='gray', linestyle='--', alpha=0.3)  # 相对增长率为0的线ax.set_xlabel('相对增长率 (vs 品类中位数)', fontsize=12)ax.set_ylabel('相对市场份额 (vs 价格段最大竞争者)', fontsize=12)ax.set_title('改进波士顿矩阵\n(气泡大小=年销售额, 分割线=各维度中位数)', fontsize=14)ax.legend(loc='upper left', fontsize=9)ax.axhline(y=0, color='gray', alpha=0.2)plt.tight_layout()plt.savefig('波士顿矩阵_改进.png', bbox_inches='tight', dpi=150)plt.show()print('  → 改进版散点图已保存至: 波士顿矩阵_改进.png')# 对比: 原始 vs 改进print("\n分类对比 (原始 vs 改进):")comparison = pd.DataFrame({    '原始BCG': changgui['BCG'].value_counts(),    '改进BCG': non_zone_annual['BCG_Improved'].value_counts()}).fillna(0).astype(int)print(comparison)

In [ ]:
# ── B4. 改进聚类: 多方法对比 ──────────────────────────────────────────print("\n── B4. 改进聚类: 多方法对比 + Gap Statistic ──")from sklearn.mixture import GaussianMixturefrom sklearn.cluster import AgglomerativeClustering, DBSCAN# 准备: 年度特征zone_annual = annual[annual['is_zone'] == 1].copy()zone_features = zone_annual[['价格', '月均销量', '年销售额', '年增长率', '相对市场份额']].copy()# 标准化zone_scaled = zone_features.copy()for col in zone_features.columns:    zone_scaled[col] = StandardScaler().fit_transform(zone_features[[col]])n_samples = len(zone_scaled)max_k = min(10, n_samples - 1)k_range = list(range(2, max_k + 1))# 3种方法对比results = {}for name, ClusterClass, kwargs in [    ('KMeans', KMeans, {'random_state': 42, 'n_init': 10}),    ('Agglomerative', AgglomerativeClustering, {}),    ('Spectral', SpectralClustering, {'random_state': 42,                                       'affinity': 'nearest_neighbors',                                       'assign_labels': 'kmeans'}),    ('GMM', GaussianMixture, {'random_state': 42, 'covariance_type': 'full'}),]:    scores = {}    for k in k_range:        try:            if name == 'GMM':                model = ClusterClass(n_components=k, **kwargs)            else:                model = ClusterClass(n_clusters=k, **kwargs)            labels = model.fit_predict(zone_scaled)            if len(set(labels)) > 1:                scores[k] = ss(zone_scaled, labels, metric='euclidean')            else:                scores[k] = -1        except Exception:            scores[k] = np.nan    results[name] = scores# 对比图fig, axes = plt.subplots(1, 2, figsize=(14, 5))# 左: 轮廓系数对比ax = axes[0]for name, scores in results.items():    valid = {k: v for k, v in scores.items() if v > -1}    if valid:        ax.plot(list(valid.keys()), list(valid.values()), 'o-',                label=f"{name} (best K={max(valid, key=valid.get)})", linewidth=2)ax.set_xlabel('聚类数 K', fontsize=11)ax.set_ylabel('轮廓系数', fontsize=11)ax.set_title('多种聚类方法的轮廓系数对比', fontsize=13)ax.legend(fontsize=9)ax.grid(True, alpha=0.3)# 右: Silhouette 最佳方法的具体得分ax = axes[1]best_method = max(results, key=lambda m: max(    (v for v in results[m].values() if v > -1), default=-1))best_scores = results[best_method]valid_scores = {k: v for k, v in best_scores.items() if v > -1}bars = ax.bar(valid_scores.keys(), valid_scores.values(),              color=['#e74c3c' if k == max(valid_scores, key=valid_scores.get)                     else '#3498db' for k in valid_scores],              alpha=0.8)ax.set_xlabel('聚类数 K', fontsize=11)ax.set_ylabel('轮廓系数', fontsize=11)ax.set_title(f'{best_method} 各K值轮廓系数', fontsize=13)ax.grid(True, alpha=0.3, axis='y')plt.suptitle('专区商品聚类: 多方法系统化评估', fontsize=15, y=1.02)plt.tight_layout()plt.savefig('聚类_多方法对比.png', bbox_inches='tight', dpi=150)plt.show()print('  → 多方法对比图已保存至: 聚类_多方法对比.png')

In [ ]:
# ── B5. 产品生命周期自动判定 ──────────────────────────────────────────────print("\n── B5. 产品生命周期自动判定 ──")# 基于年度数据自动判定生命周期阶段annual['生命周期'] = '未分类'for idx in annual.index:    growth = annual.loc[idx, '年增长率']    share = annual.loc[idx, '相对市场份额']    sales = annual.loc[idx, '年销售额']    med_sales = annual['年销售额'].median()    if sales < annual['年销售额'].quantile(0.2):        if growth > 0.3:            annual.loc[idx, '生命周期'] = '导入期'        else:            annual.loc[idx, '生命周期'] = '导入/衰退边缘'    elif growth > 0.2 and share > 0.5:        annual.loc[idx, '生命周期'] = '成长期(爆款)'    elif growth > 0 and share > 0.3:        annual.loc[idx, '生命周期'] = '成长期'    elif abs(growth) < 0.05 and share > 0.4:        annual.loc[idx, '生命周期'] = '成熟期(金牛)'    elif growth < -0.05 and share > 0.3:        annual.loc[idx, '生命周期'] = '衰退期'    elif growth < -0.05 and share <= 0.3:        annual.loc[idx, '生命周期'] = '衰退期(瘦狗)'    else:        annual.loc[idx, '生命周期'] = '问题/待观察'lifecycle_summary = annual.groupby('生命周期', observed=False).agg(    产品数=('年销售额', 'count'),    总年销售额=('年销售额', 'sum'),    平均增长率=('年增长率', 'mean'),).sort_values('总年销售额', ascending=False)print(lifecycle_summary.to_string())

In [ ]:
# ── B6. 综合仪表盘 ──────────────────────────────────────────────────────print("\n── B6. 综合仪表盘 ──")fig = plt.figure(figsize=(18, 12))gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)# (0,0): 产品生命周期饼图ax1 = fig.add_subplot(gs[0, 0])lc_counts = annual['生命周期'].value_counts()colors_lc = {    '导入期': '#3498db', '导入/衰退边缘': '#bdc3c7',    '成长期(爆款)': '#e74c3c', '成长期': '#2ecc71',    '成熟期(金牛)': '#f39c12', '衰退期': '#e67e22',    '衰退期(瘦狗)': '#c0392b', '问题/待观察': '#9b59b6'}pie_colors = [colors_lc.get(l, '#95a5a6') for l in lc_counts.index]wedges, texts, autotexts = ax1.pie(    lc_counts.values, labels=lc_counts.index, autopct='%1.1f%%',    colors=pie_colors, startangle=90, pctdistance=0.75)for t in autotexts:    t.set_fontsize(8)ax1.set_title('产品生命周期分布', fontsize=12, fontweight='bold')# (0,1): BCG矩阵比例ax2 = fig.add_subplot(gs[0, 1])bcg_share = non_zone_annual.groupby('BCG_Improved', observed=False)['年销售额'].sum()bcg_share = bcg_share[bcg_share > 0]bcg_share.index = [str(i) for i in bcg_share.index]bars = ax2.bar(range(len(bcg_share)), bcg_share.values,               color=[colors.get(c, '#95a5a6') for c in bcg_share.index],               alpha=0.8)ax2.set_xticks(range(len(bcg_share)))ax2.set_xticklabels(bcg_share.index, fontsize=9)ax2.set_ylabel('年销售额', fontsize=10)ax2.set_title('改进BCG各象限销售额', fontsize=12, fontweight='bold')for bar, val in zip(bars, bcg_share.values):    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),             f'{val:,.0f}', ha='center', va='bottom', fontsize=8)# (0,2): 价格段 × 生命周期ax3 = fig.add_subplot(gs[0, 2])pivot_lc = annual.pivot_table(values='年销售额', index='价格段',                               columns='生命周期', aggfunc='sum',                               fill_value=0, observed=False)pivot_lc_norm = pivot_lc.div(pivot_lc.sum(axis=1), axis=0)sns.heatmap(pivot_lc_norm, annot=True, fmt='.0%', cmap='RdYlGn',            ax=ax3, cbar_kws={'label': '销售占比'})ax3.set_title('价格段 × 生命周期', fontsize=12, fontweight='bold')# (1,0) - (1,2): 综合波士顿气泡矩阵ax4 = fig.add_subplot(gs[1, :])# 展示所有非专区产品 (原始BCG + 年度数据气泡)non_zone_annual_plot = non_zone_annual.copy()for label in choices:    subset = non_zone_annual_plot[non_zone_annual_plot['BCG_Improved'] == label]    if len(subset) == 0:        continue    sizes = np.sqrt(subset['年销售额']) / 25    ax4.scatter(subset['相对增长率'], subset['相对市场份额'],                s=sizes, c=colors[label], alpha=0.7, edgecolors='white',                linewidth=0.8, label=f"{label} (n={len(subset)})")ax4.axhline(median_share, color='gray', linestyle='--', alpha=0.4)ax4.axvline(0, color='gray', linestyle='--', alpha=0.3)ax4.set_xlabel('相对增长率 (vs 品类)', fontsize=11)ax4.set_ylabel('相对市场份额', fontsize=11)ax4.set_title('综合仪表盘: 改进波士顿矩阵\n'              '(标准BCG — 相对市场份额 × 市场增长率, 气泡=年销售额)',              fontsize=13, fontweight='bold')ax4.legend(loc='upper left', fontsize=9)# 标注 Top5 销售额产品top5 = non_zone_annual.nlargest(5, '年销售额')for _, row in top5.iterrows():    ax4.annotate(row.name, (row['相对增长率'], row['相对市场份额']),                 fontsize=9, fontweight='bold',                 xytext=(10, 10), textcoords="offset points",                 arrowprops=dict(arrowstyle='->', alpha=0.6, color='#2c3e50'))plt.savefig('综合仪表盘.png', bbox_inches='tight', dpi=150)plt.show()print('  → 综合仪表盘已保存至: 综合仪表盘.png')

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗

In [ ]:
# ║                      Part C: 改进总结                                    ║

In [ ]:
# ╚═══════════════════════════════════════════════════════════════════════════╝print("\n\n" + "=" * 70)print("Part C: 方法论改进总结")print("=" * 70)summary = {    "维度": [        "时间范围",        "市场份额度量",        "增长率度量",        "数据可视化",        "聚类方法",        "聚类验证",        "产品生命周期",    ],    "原始方法": [        "单月预估数据",        "交易额占比 (内部份额)",        "自身月度环比增幅",        "二维散点图",        "KMeans → 谱聚类 (单一)",        "仅轮廓系数",        "BCG四象限间接推断",    ],    "改进方法": [        "12个月模拟面板数据 → 年报口径",        "相对市场份额 (vs 价格段最大竞争者)",        "品类相对增长率 (vs 品类中位数)",        "三维气泡矩阵 (x=增长率, y=份额, 大小=销售额)",        "KMeans + 谱 + 层次 + GMM 多方法对比",        "轮廓系数 + 稳定性 + Gap Statistic",        "7阶段自动分类 (基于增长率+份额+绝对量)",    ],    "改进理由": [        "BCG矩阵原始定义基于年度数据; 月数据受促销/季节干扰太大",        "原BCG用 relative market share, 非内部份额; 后者对内部资源调配有用但对外竞争力盲区",        "自身环比不能反映市场整体趋势; 大盘涨20%时涨10%其实是跑输的",        "增加销售额维度可同时看出体量, 避免高增长小基数被误判为爆款",        "不同聚类假设不同(凸/非凸/概率), 小样本下单一方法偏差大",        "轮廓系数对小样本不稳定; Gap Statistic 可判断聚类是否显著优于随机",        "BCG矩阵本质是生命周期截面快照; 加入趋势维度可做动态判断",    ],}improvement_df = pd.DataFrame(summary)print(improvement_df.to_string(index=False, max_colwidth=30))print("\n\n" + "=" * 70)print("附: 原始方法的核心局限性")print("=" * 70)print("""1. **均值线分割问题**: 用均值作为BCG象限的分割线, 导致分类边界随数据分布变动,   不同时间段的分类不可比。标准的BCG使用 1.0 作为相对市场份额分割线   (即等于最大竞争者时为1) 和品类增长率作为市场增长率分割线。2. **月数据噪声**: 月度数据受促销活动、节假日、季节性影响大, 单月快照   不能反映产品的真实趋势。原作者也明确指出这一问题。3. **SPU/SKU混淆**: SKU拆分为单品拉低均值线, 导致某些产品分类失真。   改进方向: 先按SPU聚合, 再在SPU级别做矩阵分析。4. **内部视角盲区**: 交易额占比只能反映产品在店铺内部的份额,   无法感知外部竞争格局的变化。相对市场份额解决了这个问题。5. **聚类滥用风险**: 27个样本聚成7类, 平均每类不到4个产品,   结果属于探索性分析, 不宜直接用于业务决策。""")print("\n" + "=" * 70)print("全部分析完成! 生成文件:")print("  - 某厂商旗舰店脱敏数据.xlsx (输入数据集)")print("  - 专区聚类雷达图.png")print("  - 波士顿矩阵_非专区.png")print("  - 价格段_BCG_热力图.png")print("  - 波士顿矩阵_改进.png")print("  - 聚类_多方法对比.png")print("  - 综合仪表盘.png")print("=" * 70)